In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pendulum, re, tempfile
import os,sys
from tqdm import tqdm

In [ ]:
CAMERA_TYPE = 'S1'
RA_WINDOW = 3
RAWFILE_MAGNITUDE = 12

index_prefix = 'WLD210_S1_'

# STATUS

In [ ]:
def read_status_file(file_path,index_prefix):
    # function to read status data in either CN or EN
    def status_reader(file_path,index_prefix,has_chinese_text):
        # determine parameters according to the language
        if has_chinese_text:
            colspecs = [(0,3),(3,12),(12,25),(27,-1)]
            datetime_format = '%d.%m.%y %H:%M:%S:%f'
        else:
            colspecs = [(0,5),(7,15),(16,28),(31,-1)] 
            datetime_format = '%m/%d/%y %H:%M:%S:%f'
        colnames = ['type', 'date','time', 'message']
        
        # read the file
        data = pd.read_fwf(file_path,names =colnames, colspecs=colspecs)
        if data.empty:
            return None
        
        data['blank_id'] = index_prefix+"_"+data.index.map(str).map(lambda n: n.zfill(RAWFILE_MAGNITUDE))
        # data['blank_id'] = index_prefix+'_'+convert_local_tz(data_interval_start).format('YYYYMMDD')+"_"+data.index.map(str).map(lambda n: n.zfill(RAWFILE_MAGNITUDE))
        data.dropna(inplace=True)
        data['datetime']= data.date + ' '  + data.time
        data['datetime'] = pd.to_datetime(data['datetime'], dayfirst=True, format=datetime_format, errors = 'raise')
        return data

    #instantiate temp files to separate the rows from different language
    temp_en = tempfile.NamedTemporaryFile(delete=False)
    temp_cn = tempfile.NamedTemporaryFile(delete=False)

    # class Temp:
    #     def __init__(self, name) -> None:
    #         self.name = name 
    # temp_en = Temp('temp_en')
    # temp_cn = Temp('temp_cn')

    #determine the language of the each row of the file
    with open(file_path,'r', errors='ignore') as source, open(temp_en.name,'w') as en, open(temp_cn.name,'w') as cn:
        lines = source.readlines()
        for line in lines:
            if re.search(r'[\u4e00-\u9fff]+', line):
                cn.write(line)
            else:
                en.write(line)

    data_en = status_reader(temp_en,index_prefix, has_chinese_text=False)
    data_cn = status_reader(temp_cn,index_prefix, has_chinese_text=True)
    return pd.concat([data_en,data_cn]).sort_values(by='datetime')

In [ ]:
def read_status(status_file_path,index_prefix=index_prefix):
    data = read_status_file(status_file_path,index_prefix=index_prefix)

    #data = pd.concat([last_cycle_data,data])

    camera_type='S1'
    camera_id =42101
    data['cycle_id'] = data.message.str.extract(r"Cycle ID: (\d+)|å¾ªçŽ¯åœ°å€: (\d+)")[0].fillna(data.message.str.extract(r"Cycle ID: (\d+)|å¾ªçŽ¯åœ°å€: (\d+)")[1])
    data['camera_id'] = camera_id

    # Extract the seam number and the location. We need to separate because sometimes the location is missing and the regex r"N=(\d+):(\d+)mm" wont work
    data['seam_number']= data.message.str.extract(r"N=(\d+)")
            
    if(camera_type in ['S2L','S2U']):
        # MC 2023/11/16 change to also consider another condition when '(#21060) N=2:174mm; Concav/Convex, Start of tracking line too close to the seamp'
        data['error_name'] = data['message'].str.extract(r"Error:(.+)|\(#.*;(.*)", expand=False).fillna('').max(axis=1)
        data['error_location'] = data.message.str.extract(r":(\d+)mm")

    elif(camera_type =='S1'):
        data['error_name'] = data.message.str.extract(r"mm: ([\w\s]+)")
        data['error_location'] = data.message.str.extract(r",.(\d+)mm")

    # ST 2023/12/22 We consider global error to have the 'global' keyword inside the message
    data['error_type']=np.nan
    data.loc[data.message.str.contains(r'(?i)global.*:'),'error_type'] = 'GLOBAL Error'

    data.loc[(data['error_location'].notna() & data['error_type'].isna()),'error_type']='LOCAL Error'

    data['cycle_id'].ffill(inplace = True)
    cycle_data = data.loc[data.message.str.startswith(('Cycle','å¾ªçŽ¯åœ°å€'))].copy()


    # MC 2023/11/16 change to np.nan so that status_data['error_name'].fillna can work later
    data['error_name'] = data['error_name'].replace('', np.nan)


    #add cycleID records
    status_data = data.loc[(data['type'].isin(['NOK','ERROR','WARN','ä¸åˆæ ¼','é”™è¯¯','è­¦å‘Š']))].copy()

    status_data['error_name'].fillna(status_data.message, inplace = True)

    # MC 2023/11/16 seems not needed
    #add global error
    # status_data.loc[(status_data.error_location.isna())&(status_data['type'].isin(['NOK','ä¸åˆæ ¼']))&(~status_data['error_name'].str.startswith(('Cycle','å¾ªçŽ¯åœ°å€')))&(~status_data['message'].str.startswith(('system error','ç³»ç»Ÿé”™è¯¯'))),'error_type']='GLOBAL error'
    #add system error
    status_data.loc[status_data['message'].str.contains('system error|ç³»ç»Ÿé”™è¯¯'),'error_type']='system error'


    status_data.rename({'datetime':'local_timestamp'},axis=1,inplace=True)

    status_data['source_date'] = '' #convert_local_tz(data_interval_start).format("YYYYMMDDHH")
    status_data['source_file_path'] ='path/to/azure/blob'
    status_data = status_data[['blank_id','cycle_id','camera_id','local_timestamp','seam_number','type','error_location','error_type','error_name','source_file_path','source_date']]

    status_data['etl_processed_on'] = pendulum.now().to_iso8601_string()

    return cycle_data, status_data

# SOUVRES

In [ ]:
def read_souvres(souvres_file_path, cycle_data):

    table_line_idx = []
    fp = open(souvres_file_path)

    for i, line in enumerate(fp):
        if i==1:
            continue
        if line.find('date')>-1:
            table_line_idx.append(i)
    num_lines = i
    fp.close()
    num_tables = len(table_line_idx)
    # print(num_tables)
    # if (num_tables==0): # file is empty we stop the ETL
    #     return
    data = pd.read_fwf(souvres_file_path,header =0, skiprows =table_line_idx, parse_dates = True )
    # cycle_data = pd.read_csv(f'cycle_data.csv')
    data = data.loc[data.date!='date']
    data.drop_duplicates(inplace = True)
    data['datetime'] = data.date+' '+data.time
    try:
        data['datetime'] = pd.to_datetime(data['datetime'], dayfirst=True,format ='%m/%d/%y %H:%M:%S:%f', errors = 'raise')
    except:
        data['datetime'] = pd.to_datetime(data['datetime'], dayfirst=True,format ='%d.%m.%y %H:%M:%S:%f', errors = 'raise')

    cycle_data['datetime'] = pd.to_datetime(cycle_data['datetime'], errors = 'raise')
    #sort data in order to merge with merge_asof
    cycle_data = cycle_data.sort_values(by='datetime')
    data = data.sort_values(by='datetime')
    #merge measure data with the cycle data by taking the nearest previous matching datetime
    data = pd.merge_asof(data, cycle_data[['datetime','cycle_id']], on="datetime")
    #sort data in order to merge with merge_asof
    data = data.sort_values(by='datetime')

    data['gapw'] = data['gapw'].astype(float)
    data['ctr'] = data['ctr'].astype(float)

    return data

# Get data

In [ ]:
month = '202406' ## change here for month in data/
day = '12' ## change here for day in data/[yearmm]
root = os.path.join('data', month)
status_file_path = os.path.join(root, f'status{day}.log')

souvres_dir = os.path.join(root, f'day={day}')
souvres_file_paths = [os.path.join(souvres_dir, file) for file in os.listdir(souvres_dir)]

cycle_data, _ = read_status(status_file_path)  ## !!!!!!!! maybe buggy due to unknown reason. if bug, use the code below instead where we read from archived csv of status data
# cycle_data = pd.read_csv('data/cycle_data04.csv')

signals = []
for souvres_file_path in tqdm(souvres_file_paths):
    data = read_souvres(souvres_file_path, cycle_data=cycle_data)
    # if len(data)>10000:
    #     break
    
    for i, cycle in enumerate(data['cycle_id'].unique()):
        
        sample = data.loc[data.cycle_id==cycle]
        
        # magnitude = sample['rolling_gapw'].max()-sample['rolling_gapw'].min()
        sample_length = len(sample)
        for j, image in enumerate(sample['image'].unique()):
            # plt.figure(i*sample_length +j)
            image_sample = sample.loc[sample.image == image]
            signal_class = image_sample[["ctr", "pos"]].to_numpy().astype(float)
            
            signals.append(signal_class)


In [ ]:
len(signals)

In [ ]:
import pickle


# Save array to pickle file
with open('data/pos20240612.pkl', 'wb') as f:
    pickle.dump(signals, f)

In [ ]:
## read data from pickle for validation

# with open('data/pos20240703.pkl', 'rb') as f:
#     signals_pos = pickle.load(f)

# print(len(signals_pos))
# print(signals_pos[:5])

# Read souvres logs without status file

This section reads camera measurement logs directly from `souvres*.log`. It creates an artificial `product_id` by counting a new product every time a new seam block starts with image `00010101`.


In [ ]:
def read_souvres_file_without_status(file_path, start_product_id=1, cycle_start_image='00010101'):
    """Read one souvres log file and assign artificial product_id values.

    A seam is identified by each repeated header block beginning with
    "date time trigger image ...". A new product starts when a new seam block
    has image == cycle_start_image, for example "00010101".
    """
    columns = [
        'date', 'time', 'trigger', 'image', 'ctr', 'pos', 'focus', 'gapw',
        'hdiff', 'strgh', 'abwlf', 'abwrt', 'abblf', 'abbrt', 'errors',
        'warnings', 'exc', 'debug'
    ]
    numeric_columns = [
        'ctr', 'pos', 'focus', 'gapw', 'hdiff', 'strgh',
        'abwlf', 'abwrt', 'abblf', 'abbrt'
    ]

    rows = []
    seam_block_id = -1

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as fp:
        for line in fp:
            line = line.strip()
            if not line:
                continue

            # Every header line marks the beginning of a new seam data block.
            if line.lower().startswith('date'):
                seam_block_id += 1
                continue

            parts = line.split()
            if len(parts) != len(columns):
                continue

            row = dict(zip(columns, parts))
            row['seam_block_id'] = seam_block_id
            row['source_file_path'] = str(file_path)
            rows.append(row)

    data = pd.DataFrame(rows)
    if data.empty:
        return data

    data['datetime'] = pd.to_datetime(
        data['date'] + ' ' + data['time'],
        format='%d.%m.%y %H:%M:%S:%f',
        errors='coerce'
    )

    for col in numeric_columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    block_images = data.groupby('seam_block_id', sort=False)['image'].first()
    product_id = start_product_id - 1
    block_to_product = {}

    for block_id, image in block_images.items():
        if product_id < start_product_id or image == cycle_start_image:
            product_id += 1
        block_to_product[block_id] = product_id

    data['product_id'] = data['seam_block_id'].map(block_to_product)
    data['seam_number'] = data.groupby('product_id')['seam_block_id'].rank(method='dense').astype(int)

    return data


def read_souvres_folder_without_status(root_folder='data/202605', cycle_start_image='00010101'):
    """Read all souvres*.log files below a month folder and return one DataFrame.

    The product_id is calculated globally after concatenating all files, so the
    counter does not restart at each file boundary.
    """
    root_folder = os.fspath(root_folder)
    file_paths = []

    for day_name in sorted(os.listdir(root_folder)):
        day_path = os.path.join(root_folder, day_name)
        if not os.path.isdir(day_path):
            continue

        for file_name in sorted(os.listdir(day_path)):
            if file_name.lower().startswith('souvres') and file_name.lower().endswith('.log'):
                file_paths.append(os.path.join(day_path, file_name))

    data_frames = []
    for file_path in tqdm(file_paths):
        file_data = read_souvres_file_without_status(
            file_path,
            start_product_id=1,
            cycle_start_image=cycle_start_image
        )
        if file_data.empty:
            continue
        data_frames.append(file_data)

    if not data_frames:
        return pd.DataFrame()

    data = pd.concat(data_frames, ignore_index=True)

    block_keys = ['source_file_path', 'seam_block_id']
    block_images = data.groupby(block_keys, sort=False)['image'].first().reset_index()

    product_id = 0
    block_to_product = {}
    for _, block in block_images.iterrows():
        if product_id == 0 or block['image'] == cycle_start_image:
            product_id += 1
        block_key = (block['source_file_path'], block['seam_block_id'])
        block_to_product[block_key] = product_id

    data['product_id'] = [
        block_to_product[(source_file_path, seam_block_id)]
        for source_file_path, seam_block_id in zip(data['source_file_path'], data['seam_block_id'])
    ]

    block_product_map = block_images.copy()
    block_product_map['product_id'] = [
        block_to_product[(source_file_path, seam_block_id)]
        for source_file_path, seam_block_id in zip(
            block_product_map['source_file_path'],
            block_product_map['seam_block_id']
        )
    ]
    block_product_map['seam_number'] = block_product_map.groupby('product_id').cumcount() + 1
    seam_number_map = {
        (source_file_path, seam_block_id): seam_number
        for source_file_path, seam_block_id, seam_number in zip(
            block_product_map['source_file_path'],
            block_product_map['seam_block_id'],
            block_product_map['seam_number']
        )
    }
    data['seam_number'] = [
        seam_number_map[(source_file_path, seam_block_id)]
        for source_file_path, seam_block_id in zip(data['source_file_path'], data['seam_block_id'])
    ]

    return data

In [ ]:
import pickle

# Example: read one file
one_file_data = read_souvres_file_without_status('data/202605/day=01/souvres00.log')
print(one_file_data.shape)
one_file_data.head()


In [13]:

# Example: read all souvres logs in data/202605 and save pos signals by product + date + seam
all_souvres_data = read_souvres_folder_without_status('data/202605')
all_souvres_data['date'] = all_souvres_data['datetime'].dt.date


100%|██████████| 692/692 [00:54<00:00, 12.78it/s] 


In [14]:

pos_signal_by_product_date_and_seam = {}
for (product_id, date, seam_number), group in all_souvres_data.groupby(['product_id', 'date', 'seam_number'], sort=False):
    pos_signal_by_product_date_and_seam.setdefault(product_id, {}).setdefault(date, {})[seam_number] = group['pos'].to_numpy()

with open('data/pos_signal_202605_by_product_date_and_seam.pkl', 'wb') as f:
    pickle.dump(pos_signal_by_product_date_and_seam, f)

print(all_souvres_data.shape)
all_souvres_data.head()

(6183171, 23)


,date,time,trigger,image,ctr,pos,focus,gapw,hdiff,strgh,...,abbrt,errors,warnings,exc,debug,seam_block_id,source_file_path,datetime,product_id,seam_number
0,2026-05-01,00:15:59:827,00000001,00010101,1,81,0,0,104,0,...,43,80000000,00000000,00000000,44040000,1,data/202605\day=01\souvres00.log,2026-05-01 00:15:59.827,1,1
1,2026-05-01,00:15:59:837,00010002,00010101,2,92,0,0,97,0,...,51,80000000,00000000,00000000,44040000,1,data/202605\day=01\souvres00.log,2026-05-01 00:15:59.837,1,1
2,2026-05-01,00:15:59:845,00010003,00010101,3,100,0,0,120,0,...,51,80000000,00000000,00000000,44040000,1,data/202605\day=01\souvres00.log,2026-05-01 00:15:59.845,1,1
3,2026-05-01,00:15:59:855,00010004,00010101,4,107,0,0,97,0,...,54,80000000,00000000,00000000,44040000,1,data/202605\day=01\souvres00.log,2026-05-01 00:15:59.855,1,1
4,2026-05-01,00:15:59:865,00010005,00010101,5,108,0,0,112,0,...,50,80000000,00000000,00000000,44040000,1,data/202605\day=01\souvres00.log,2026-05-01 00:15:59.865,1,1
